# MaterialGAN batch pipeline on Colab

Runs `batch_run.py` over a Drive-hosted dataset of 100+ object folders. Each object folder must contain a `raw/` subfolder with photos that include the AprilTag markers.

Outputs (texture maps, re-renders, optional `vid.gif`) are written **directly into each object folder on Drive** as they're produced — nothing is held in Colab's ephemeral storage. If Colab disconnects, just reconnect and re-run; finished folders are skipped automatically.

**Before running:** set runtime to GPU (`Runtime > Change runtime type > A100 / H100` if available).

## 1. Sanity check the GPU

In [ ]:
!nvidia-smi

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configure paths and capture params

**Edit `DATASET_DRIVE_PATH`** to point at the Drive folder that directly contains your 100+ object folders. Each object folder should look like:

```
<DATASET_DRIVE_PATH>/keychain/raw/{1.jpg, 2.jpg, ...}
<DATASET_DRIVE_PATH>/bracelet/raw/{1.jpg, 2.jpg, ...}
...
```

If your photos are loose in each object folder instead of inside `raw/`, run the optional "normalize layout" cell below.

In [ ]:
from pathlib import Path

# --- EDIT THIS ---
DATASET_DRIVE_PATH = Path('/content/drive/MyDrive/reflective_dataset')  # <- your object folders live directly under here

# Test-run toggle. Set to a folder name (e.g. 'zippo') to only process that one.
# Set to None to process the whole dataset. Set to comma-separated names ('zippo,key') for multiple.
TEST_ONLY = 'zippo'
# -----------------

SIZE     = 17.0   # AprilTag print size in cm (red-arrow measurement from README)
DEPTH    = 0.1    # distance between marker plane and material plane in cm
DO_ENVMAP = False # set True if you want the relighting vid.gif per object (slower)

# --- Pivotal Tuning (PTI) ---
# After latent optimization, fine-tune the MaterialGAN generator weights to each
# surface. Helps on out-of-distribution materials (e.g. specular metals) that the
# frozen prior can't represent with a latent alone. Adds ~1-2 min/surface on GPU.
USE_PTI    = True  # set True to enable per-surface generator fine-tuning
PTI_EPOCHS = 300    # PTI steps; lower (~150) if maps look over-sharpened / noisy
PTI_LR     = 3e-4   # PTI learning rate; lower (1e-4) to be more conservative

REPO_URL = 'https://github.com/currahhee/meterialgan.git'
REPO_DIR = Path('/content/meterialgan')
DRIVE_CKP = Path('/content/drive/MyDrive/materialgan_ckp')   # pretrained weights cached here so they don't redownload
LOG_PATH  = Path('/content/drive/MyDrive/materialgan_log.json')

assert DATASET_DRIVE_PATH.exists(), f'Dataset path not found: {DATASET_DRIVE_PATH}'
print('Dataset:', DATASET_DRIVE_PATH)
print('Subfolders:', sum(1 for _ in DATASET_DRIVE_PATH.iterdir() if _.is_dir()))
print('TEST_ONLY:', TEST_ONLY if TEST_ONLY else '(none — full dataset)')
print('PTI:', f'ON (epochs={PTI_EPOCHS}, lr={PTI_LR})' if USE_PTI else 'off')


## 4. Clone repo + cache model weights on Drive

Pretrained MaterialGAN weights (~hundreds of MB) get downloaded on first run; storing them on Drive avoids redownloading every session.

In [ ]:
import os, shutil

if REPO_DIR.exists():
    print('Repo exists, pulling latest...')
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

DRIVE_CKP.mkdir(parents=True, exist_ok=True)
repo_ckp = REPO_DIR / 'ckp'

# Seed Drive with anything that shipped in repo/ckp (e.g. placeholder file), then symlink.
if repo_ckp.exists() and not repo_ckp.is_symlink():
    for item in repo_ckp.iterdir():
        dest = DRIVE_CKP / item.name
        if not dest.exists():
            shutil.copy2(item, dest) if item.is_file() else shutil.copytree(item, dest)
    shutil.rmtree(repo_ckp)
if repo_ckp.is_symlink():
    repo_ckp.unlink()
os.symlink(DRIVE_CKP, repo_ckp, target_is_directory=True)

%cd {REPO_DIR}
!ls -la ckp

## 5. Install dependencies

In [ ]:
!pip install -q opencv-python==4.11.0.86 matplotlib==3.10.1 tqdm==4.67.1 requests==2.32.4 pupil-apriltags==1.0.4.post11 mitsuba==3.6.4
import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 5b. Drop in `batch_run.py`

The cloned fork doesn't yet contain `batch_run.py` (only your local copy does). This cell writes the batch driver into the repo on Colab. Once you push the file to your fork, this cell becomes a no-op (overwriting with identical content).

In [ ]:
%%writefile /content/meterialgan/batch_run.py
# -*- coding: utf-8 -*-
"""Batch MaterialGAN pipeline. See colab_materialgan.ipynb for usage.

Each object folder is processed in a *separate subprocess* to isolate any
native-library (OpenCV / pupil-apriltags) heap corruption from one folder
contaminating the next.
"""
import argparse, json, os, subprocess, sys, time, traceback
from pathlib import Path

COMPLETION_MARKER = Path('optim_latent') / '1024' / 'dif.png'
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}
REPO_DIR = Path('/content/meterialgan')

WORKER_TEMPLATE = """
import sys
sys.path.insert(0, {repo!r})
from pathlib import Path
from src.scripts import (
    gen_targets_from_capture, optim_ganlatent, optim_perpixel, render_envmap,
)
folder = Path({folder!r})
gen_targets_from_capture(folder, size={size}, depth={depth})
optim_ganlatent(folder / 'optim_latent_256.json', 256, 0.02, [1000, 10, 10], tex_init='auto', pti={pti}, pti_epochs={pti_epochs}, pti_lr={pti_lr})
optim_perpixel(folder / 'optim_pixel_256_to_512.json', 512, 0.01, 20, tex_init='textures')
optim_perpixel(folder / 'optim_pixel_512_to_1024.json', 1024, 0.01, 20, tex_init='textures')
if {do_envmap}:
    render_envmap(folder / 'optim_latent' / '1024', 256)
"""

def has_raw_images(folder):
    raw = folder / 'raw'
    return raw.is_dir() and any(p.suffix.lower() in IMAGE_SUFFIXES for p in raw.iterdir())

def is_done(folder):
    return (folder / COMPLETION_MARKER).exists()

def process_one(folder, size, depth, do_envmap, pti, pti_epochs, pti_lr):
    code = WORKER_TEMPLATE.format(
        repo=str(REPO_DIR), folder=str(folder),
        size=size, depth=depth, do_envmap=bool(do_envmap),
        pti=bool(pti), pti_epochs=int(pti_epochs), pti_lr=float(pti_lr),
    )
    r = subprocess.run(
        [sys.executable, '-u', '-c', code],
        cwd=str(REPO_DIR),
    )
    if r.returncode != 0:
        raise RuntimeError(f'worker subprocess exited with code {r.returncode}')

def main():
    p = argparse.ArgumentParser()
    p.add_argument('--data-root', required=True, type=Path)
    p.add_argument('--size', type=float, default=17.0)
    p.add_argument('--depth', type=float, default=0.1)
    p.add_argument('--envmap', action='store_true')
    p.add_argument('--pti', action='store_true', help='Fine-tune the generator per surface (Pivotal Tuning).')
    p.add_argument('--pti-epochs', type=int, default=300, help='PTI steps (default 300).')
    p.add_argument('--pti-lr', type=float, default=3e-4, help='PTI learning rate (default 3e-4).')
    p.add_argument('--log', type=Path, default=None)
    p.add_argument('--limit', type=int, default=None)
    p.add_argument('--only', type=str, default=None)
    args = p.parse_args()

    if not args.data_root.is_dir():
        print(f'ERROR: --data-root {args.data_root} is not a directory', file=sys.stderr)
        return 2
    only_set = set(s.strip() for s in args.only.split(',')) if args.only else None

    candidates = sorted(d for d in args.data_root.iterdir() if d.is_dir())
    folders = [d for d in candidates if has_raw_images(d)]
    if only_set is not None:
        folders = [d for d in folders if d.name in only_set]
    if args.limit is not None:
        folders = folders[:args.limit]

    skipped_no_raw = [d.name for d in candidates if d not in folders and (only_set is None or d.name in only_set)]
    if skipped_no_raw:
        print(f'Note: {len(skipped_no_raw)} folder(s) skipped (no raw/ images): {skipped_no_raw[:10]}{"..." if len(skipped_no_raw) > 10 else ""}')
    print(f'Found {len(folders)} folder(s) to consider under {args.data_root}')
    print(f'Params: size={args.size} cm, depth={args.depth} cm, envmap={args.envmap}, pti={args.pti} (epochs={args.pti_epochs}, lr={args.pti_lr})\n')

    results = []
    t_start = time.time()
    for i, folder in enumerate(folders, 1):
        t0 = time.time()
        header = f'[{i}/{len(folders)}] {folder.name}'
        print('=' * len(header)); print(header); print('=' * len(header))
        if is_done(folder):
            print('  -> already complete, skipping')
            results.append({'folder': folder.name, 'status': 'skipped', 'elapsed_s': 0.0})
            continue
        try:
            process_one(folder, args.size, args.depth, args.envmap, args.pti, args.pti_epochs, args.pti_lr)
            elapsed = time.time() - t0
            print(f'  -> OK ({elapsed:.1f}s)')
            results.append({'folder': folder.name, 'status': 'ok', 'elapsed_s': round(elapsed, 1)})
        except Exception as e:
            elapsed = time.time() - t0
            tb = traceback.format_exc()
            print(f'  -> FAILED after {elapsed:.1f}s: {e}'); print(tb)
            results.append({'folder': folder.name, 'status': 'failed', 'elapsed_s': round(elapsed, 1), 'error': str(e), 'trace': tb})
        if args.log:
            args.log.parent.mkdir(parents=True, exist_ok=True)
            args.log.write_text(json.dumps(results, indent=2))

    total = time.time() - t_start
    ok = sum(1 for r in results if r['status'] == 'ok')
    skipped = sum(1 for r in results if r['status'] == 'skipped')
    failed = sum(1 for r in results if r['status'] == 'failed')
    bar = '=' * 60
    print(f'\n{bar}\nSUMMARY: {ok} processed | {skipped} skipped | {failed} failed | total {total/60:.1f} min\n{bar}')
    if failed:
        print('Failed folders:')
        for r in results:
            if r['status'] == 'failed':
                print(f"  - {r['folder']}: {r['error']}")
    return 0 if failed == 0 else 1

if __name__ == '__main__':
    sys.exit(main())


## 6. (Optional) normalize folder layout

MaterialGAN's `gen_targets_from_capture` expects photos under `<object>/raw/*`. If your photos sit loose directly inside each object folder, this cell moves them into `raw/`. **Safe to skip** if your layout is already correct.

In [ ]:
import shutil

IMG_SUFFIXES = {'.jpg', '.jpeg', '.png'}
moved_total = 0
for obj_folder in sorted(p for p in DATASET_DRIVE_PATH.iterdir() if p.is_dir()):
    raw = obj_folder / 'raw'
    if raw.exists() and any(p.suffix.lower() in IMG_SUFFIXES for p in raw.iterdir()):
        continue
    raw.mkdir(exist_ok=True)
    loose = [p for p in obj_folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_SUFFIXES]
    for img in loose:
        shutil.move(str(img), str(raw / img.name))
        moved_total += 1
    if loose:
        print(f'{obj_folder.name}: moved {len(loose)} images into raw/')
print(f'\nTotal images moved: {moved_total}')

## 7. Dry run — list what will be processed

Quickly check the runner sees the folders correctly *before* committing H100 time.

In [ ]:
from pathlib import Path

MARKER = Path('optim_latent') / '1024' / 'dif.png'
to_run, done, no_raw = [], [], []
for d in sorted(p for p in DATASET_DRIVE_PATH.iterdir() if p.is_dir()):
    if not (d / 'raw').is_dir() or not any((d/'raw').iterdir()):
        no_raw.append(d.name)
    elif (d / MARKER).exists():
        done.append(d.name)
    else:
        to_run.append(d.name)

print(f'Will process : {len(to_run)}')
print(f'Already done : {len(done)}')
print(f'Missing raw/ : {len(no_raw)}')
print()
print('First 10 to process:', to_run[:10])

## 8. Run the batch

This is the long-running cell. Each object takes a few minutes on H100. Output streams live and per-object outputs are saved to Drive incrementally, so a disconnect at any point is safe — just re-run this cell after reconnecting.

**Debugging knobs:** add `--limit 3` to process only 3 folders, or `--only keychain,bracelet` to target specific ones.

In [ ]:
%cd {REPO_DIR}
envmap_flag = '--envmap' if DO_ENVMAP else ''
only_flag = f'--only {TEST_ONLY}' if TEST_ONLY else ''
pti_flag = f'--pti --pti-epochs {PTI_EPOCHS} --pti-lr {PTI_LR}' if USE_PTI else ''
!python -u batch_run.py \
    --data-root {DATASET_DRIVE_PATH} \
    --size {SIZE} --depth {DEPTH} \
    --log {LOG_PATH} \
    {only_flag} {envmap_flag} {pti_flag}


In [ ]:
from pathlib import Path
z = Path('/content/drive/MyDrive/reflective_dataset/zippo')
for p in sorted(z.rglob('*')):
    if p.is_file():
        print(p.relative_to(z), '|', p.stat().st_size, 'bytes')


## 9. Inspect results

In [ ]:
import json
from collections import Counter

if LOG_PATH.exists():
    log = json.loads(LOG_PATH.read_text())
    counts = Counter(r['status'] for r in log)
    print('Status counts:', dict(counts))
    failed = [r for r in log if r['status'] == 'failed']
    if failed:
        print(f'\n{len(failed)} failures:')
        for r in failed:
            print(f"  - {r['folder']}: {r['error']}")
else:
    print('No log file yet at', LOG_PATH)

In [ ]:
from IPython.display import Image, display, Markdown

PREVIEW_N = 3
completed = [d for d in sorted(DATASET_DRIVE_PATH.iterdir()) if d.is_dir() and (d / MARKER).exists()][:PREVIEW_N]
for d in completed:
    display(Markdown(f'### {d.name}'))
    out = d / 'optim_latent' / '1024'
    for name in ('tex.jpg', 'dif.png', 'nom.png', 'spe.png', 'rgh.png'):
        p = out / name
        if p.exists():
            display(Markdown(f'**{name}**'))
            display(Image(filename=str(p)))